In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_entries = Commonlib.list_files(CONFIG_BASE_PATH, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
medallion = Medallion(config)

In [0]:
# load history data from static table if target is empty
try:
    if spark.catalog.tableExists(medallion.table_lookup["gold"]):
        if spark.read.table(medallion.table_lookup["gold"]).count() == 0:
            medallion.logger.info("Table already exists in the gold layer and is empty. Load history data")
            medallion.df = spark.read.table(
                MTU.get_fully_qualified_table_name("external", "static", config["silver"]["name"])
            ).select([col["name"] for col in medallion.gold_schema])

            if config["gold"]["load_type"].lower() == "scd_type_2":
                medallion.df = medallion.df.withColumn(
                    "edw_end_tsp",
                    F.when(
                        medallion.df.edw_end_tsp == "9999-12-31T23:59:59.000+0000",
                        F.lit(None),
                    ).otherwise(medallion.df.edw_end_tsp),
                )

            medallion.write_to_delta(load_type="overwrite", layer="gold")
    else:
        raise ValueError("Table does not exist in the catalog.")
except Exception as e:
    medallion.logger.error(
        f"Error writing data from static.{medallion.name} table to gold pricing.{medallion.name}. Error message: {e}"
    )
    raise

In [0]:
try:
    # get max watermark value from gold
    gold_tbl = MTU.get_fully_qualified_table_name("commercial", "pricing", medallion.name)
    watermark_value = spark.sql(f"select max({config['gold']['watermark']}) from {gold_tbl}").collect()[0][0]
    medallion.logger.info(f"Max Watermark value from gold is {watermark_value}")

    # Get Incremental Data from silver
    silver_tbl = MTU.get_fully_qualified_table_name(
        config["silver"]["catalog"], config["silver"]["schema"], config["silver"]["name"]
    )
    df = spark.read.table(silver_tbl).filter(f"{config['gold']['watermark']} > '{watermark_value}'")

    if config["gold"]["load_type"].lower() == "scd_type_2":
        df = df.withColumn("edw_start_tsp", F.expr("mod_tsp")).withColumn("edw_end_tsp", F.lit(None))
    elif config["gold"]["load_type"].lower() != "upsert":  # only scd_type_2 and upsert are currently supported
        raise NotImplementedError(f"Load type {config['gold']['load_type']} currently not supported")

    # perform write operation
    medallion.df = df.select([col["name"] for col in medallion.gold_schema])
    medallion.write_to_delta(load_type=config["gold"]["load_type"], layer="gold")
except NotImplementedError as nie:
    medallion.logger.error(str(nie))
    raise
except Exception as e:
    medallion.logger.error(f"Error writing data into gold table {medallion.name}. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )